00 - Profiling: Replikation der ZEW-Eckwerte und Konsistenzpruefung.

Phase-0-Output. Laeuft in unter 10 Sekunden, dient als Smoke-Test der
gesamten Pipeline. Fuer Jupyter Lab: als .py mit jupytext oeffnen oder
direkt python ausfuehren.

    python notebooks/00_profiling.py

In [1]:
import sys
from pathlib import Path
_root = (
    Path(__file__).parent.parent if '__file__' in globals() else
    next(p for p in [Path.cwd(), Path.cwd().parent] if (p / 'src').exists())
)
sys.path.insert(0, str(_root))
__import__('os').chdir(_root)
from src.load import load

import pandas as pd

In [2]:
df = load('data/raw/Digitalhaushalt_Open_Data.xlsx')
print(f"Zeilen: {len(df):,}")
print(f"Jahre: {sorted(df['jahr'].unique())}")
print(f"Unique Titel-IDs: {df['id'].nunique():,}\n")

Zeilen: 21,358
Jahre: [np.int64(2019), np.int64(2021), np.int64(2023), np.int64(2024)]
Unique Titel-IDs: 6,240



In [3]:
print("=== Replikation ZEW-Eckwerte (digi_soll_eng in Mrd. EUR) ===")
agg = df.groupby('jahr').agg(
    soll_mrd=('soll', lambda x: x.sum() / 1e6),
    eng_mrd=('digi_soll_eng', lambda x: x.sum() / 1e6),
    weit_mrd=('digi_soll_weit', lambda x: x.sum() / 1e6),
    n_titel=('id', 'count'),
).round(2)
print(agg)

erwartet = {2019: 8.5, 2021: 16.6, 2023: 19.2, 2024: 17.9}
print("\nAbweichung digi_soll_eng (Replikation - erwartet):")
for j, e in erwartet.items():
    actual = agg.loc[j, 'eng_mrd']
    delta = actual - e
    print(f"  {j}: {actual:.2f} vs. {e}  Delta = {delta:+.2f} Mrd ({delta/e*100:+.1f}%)")

=== Replikation ZEW-Eckwerte (digi_soll_eng in Mrd. EUR) ===
      soll_mrd  eng_mrd  weit_mrd  n_titel
jahr                                      
2019    356.40     8.51      9.64     5015
2021    572.73    16.55     17.79     5299
2023    461.21    19.19     20.54     5486
2024    476.81    17.94     19.09     5558

Abweichung digi_soll_eng (Replikation - erwartet):
  2019: 8.51 vs. 8.5  Delta = +0.01 Mrd (+0.1%)
  2021: 16.55 vs. 16.6  Delta = -0.05 Mrd (-0.3%)
  2023: 19.19 vs. 19.2  Delta = -0.01 Mrd (-0.1%)
  2024: 17.94 vs. 17.9  Delta = +0.04 Mrd (+0.2%)


In [4]:
print("\n=== Konsistenzpruefungen ===")
viol_ew = (df['digi_soll_eng'] > df['digi_soll_weit']).sum()
# weit > soll ist trivial verletzt, wenn soll < 0 (Globale Minderausgaben).
# Pruefen nur fuer nicht-negative soll-Werte:
viol_ws = ((df['digi_soll_weit'] > df['soll']) & (df['soll'] >= 0)).sum()
dup = df.duplicated(subset=['id', 'jahr']).sum()
neg_soll = (df['soll'] < 0).sum()
print(f"  Verletzungen eng > weit: {viol_ew}")
print(f"  Verletzungen weit > soll (ohne Globale Minderausgaben): {viol_ws}")
print(f"  Doppelte (id, jahr): {dup}")
print(f"  Negative soll (Globale Minderausgaben): {neg_soll}")


=== Konsistenzpruefungen ===
  Verletzungen eng > weit: 0
  Verletzungen weit > soll (ohne Globale Minderausgaben): 0
  Doppelte (id, jahr): 0
  Negative soll (Globale Minderausgaben): 121


In [5]:
print("\n=== ID-Stabilitaet ueber Jahre ===")
id_jahr = df.groupby('id')['jahr'].nunique()
print(f"  Titel-IDs gesamt: {df['id'].nunique():,}")
for k in [4, 3, 2, 1]:
    n = (id_jahr == k).sum()
    pct = n / df['id'].nunique() * 100
    print(f"  in {k} Jahr(en): {n:,} ({pct:.1f}%)")


=== ID-Stabilitaet ueber Jahre ===
  Titel-IDs gesamt: 6,240
  in 4 Jahr(en): 4,449 (71.3%)
  in 3 Jahr(en): 511 (8.2%)
  in 2 Jahr(en): 749 (12.0%)
  in 1 Jahr(en): 531 (8.5%)


In [6]:
print("\n=== Eng-Weit-Spreizung des Gesamthaushalts ===")
sp = df.groupby('jahr').agg(
    eng=('digi_soll_eng', lambda x: x.sum() / 1e6),
    weit=('digi_soll_weit', lambda x: x.sum() / 1e6),
)
sp['delta'] = sp['weit'] - sp['eng']
sp['delta_pct'] = (sp['delta'] / sp['weit'] * 100).round(1)
print(sp.round(2))


=== Eng-Weit-Spreizung des Gesamthaushalts ===
        eng   weit  delta  delta_pct
jahr                                
2019   8.51   9.64   1.13       11.7
2021  16.55  17.79   1.25        7.0
2023  19.19  20.54   1.35        6.6
2024  17.94  19.09   1.15        6.0
